Relier la borne de tension de sortie Vout du capteur de pH à la borne de mesure EA0 de la carte Arduino.

Le programme à envoyer à Arduino est acquisition_continue.ino

On commence par importer les bibliothèques nécessaires. Attention : NE PAS EFFACER !

In [ ]:
%run Import_bibliotheques.ipynb

%run Arduino.ipynb

connexion_arduino()

In [ ]:
input("Mettre la sonde dans la solution tampon de pH 7 et appuyer sur entrée")
# Réinitialisation de l'Arduino et attente de la première ligne valide
reset_arduino()
tension7 = mesure_unique()
input("Mettre la sonde dans la solution tampon de pH 4 et appuyer sur entrée")
# Réinitialisation de l'Arduino et attente de la première ligne valide
reset_arduino()
tension4 = mesure_unique()
a = 3. * (tension7 - tension4)
b = 7. - a * tension7

In [ ]:
valeurs = ["V,pH"]
while True :
    entree = input("Entrer le volume versé en mL (entrer une ligne vide pour arrêter) : ")
    if entree.strip() == "" :
        break
    try :
        volume = float(entree.replace(',', '.'))
    except ValueError :
        print("Valeur invalide : veuillez recommencer.")
        continue
    try :
        if 'ser' not in globals() or not ser.is_open :
            connexion_arduino()
        # Réinitialisation de l'Arduino et attente de la première ligne valide
        reset_arduino()
        tension = mesure_unique()
    except Exception as erreur :
        print(f"Erreur avec l'Arduino : {erreur}")
        break
    pHactuel = a * tension + b
    print(f"pH mesuré : {pHactuel:.2f}")
    valeurs.append(f"{volume},{pHactuel}")
    with open("titragepH.csv", 'w') as fichier :
        fichier.write("\n".join(valeurs) + "\n")

Importation de tableaux enregistrés localement en fichiers csv :

In [ ]:
tableurVersVariables('titragepH.csv')

On lisse les valeurs de pH et on dérive numériquement le pH par rapport au volume versé :

In [ ]:
tck = interpolate.splrep(V, pH, s=0.003) # la fonction renvoie un tuple contenant les informations du lissage
# le paramètre 's' permet d'ajuster le lissage : s=0 la courbe passe obligatoirement par tous les points.
# on peut ajuster le lissage de sorte que la dérivée ne soit pas trop bruitée

vecV = linspace(0,max(V),1000) # On définit un vecteur de volumes régulièrement espacés pour le tracé et le calcul de la dérivée
pHLisse = interpolate.splev(vecV,tck) # On calcule le lissage des points de mesures

dervpH = gradient(pHLisse) / gradient(vecV) # On calcule la dérivée du pH par rapport au volume
dervpH = dervpH/max(dervpH)*10 # on normalise la valeur de la dérivée pour l'ajuste à l'affichage

Tracer de graphique(s) à partir des tableaux précédents :

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=V, y=pH, mode="markers", name="pH mesurés",
    marker=dict(symbol="cross", size=10, color="blue")))

fig.add_trace(go.Scatter(x=vecV, y=pHLisse, mode="lines", name="courbe lissée de pH",
    line=dict(width=2)))

fig.add_trace(go.Scatter(x=vecV, y=dervpH, mode="lines", name="dpH/dV",
    line=dict(width=2)))

# Cosmétique du graphique :
#fig.update_yaxes(scaleanchor="x", scaleratio=1)    # à décommenter pour un repère orthonormé
fig.update_xaxes(showgrid=True)    # grille verticale (par défaut)
fig.update_yaxes(showgrid=True)    # grille horizontale (par défaut)
fig.update_layout(
    showlegend=True,    # légende du graphique,
    title="Courbe de titrage",    # mettre le titre désiré entre les guillemets,
    xaxis_title="Volume versé (mL)",    # nom de l'axe des abscisses,
    yaxis_title="pH ; dpH/dV",    # nom de l'axe des ordonnées,
    height=600)    # hauteur du graphique en pixels ; ex. : 450, 700...